In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import pickle


df = pd.read_csv('Titanic-Dataset.csv')


df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)


X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'FamilySize', 'IsAlone']]
y = df['Survived']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (712, 9)
Testing shape: (179, 9)


In [3]:
from sklearn.pipeline import make_pipeline

num_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']
cat_features = ['Sex', 'Embarked']

num_transformer = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

cat_transformer = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore')
)

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

titanic_pipeline = make_pipeline(
    preprocessor,
    LogisticRegression(random_state=42)
)

titanic_pipeline.fit(X_train, y_train)

print("Pipeline successfully fitted!")

Pipeline successfully fitted!


In [4]:
import joblib
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make predictions using the fitted pipeline directly on raw test features
y_pred = titanic_pipeline.predict(X_test)

print("=== PIPELINE EVALUATION RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Save the complete pipeline (Preprocessing + Model) using Joblib
model_filename = 'titanic_pipeline.joblib'
joblib.dump(titanic_pipeline, model_filename)

print(f"\nPipeline successfully saved to {model_filename}!")

=== PIPELINE EVALUATION RESULTS ===
Accuracy: 0.8044692737430168

Confusion Matrix:
 [[97 13]
 [22 47]]

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.88      0.85       110
           1       0.78      0.68      0.73        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179


Pipeline successfully saved to titanic_pipeline.joblib!


In [ ]:
!pip install streamlit
!streamlit run app.py

